<a href="https://colab.research.google.com/github/eloapires/ExerciosDeJavaScript/blob/main/VENDAMAIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **TRATADOR VENDA MAIS**

> ***Objetivo***

* Automatizar o tratamento das planilhas da campanha VENDA MAIS, reduzindo trabalho manual e evitando erros no cadastro de faixas e preços.

                                                           O que o script faz

* Cria automaticamente a faixa inicial (0 até QTD INICIAL - 1)
* Não duplica itens sem promoção (0 a 999999)
* Preenche preços com base na tabela padrão
* Ordena corretamente as faixas por código e quantidade
* Valida incoerências (faixa maior com preço maior)
* Gera coluna de status
* Cria aba com erros para revisão


                                                            Arquivos necessários

1. **VENDA MAIS**

* CODIGO
* DESCRIÇÃO PRODUTO
* QTD INICIAL
* QTD FINAL
* PREÇO FINAL

2. **Tabela padrão**

* CODIGO
* PREÇO TABELA

**Códigos devem ser iguais nas duas planilhas e sem duplicidade na tabela padrão**

                                                                   Como usar
1. Clique em **Executar**
2. Envie os dois arquivos
3. Informe os nomes (se solicitado)
4. Baixe o arquivo gerado

> Resultado

_Arquivo Excel com:_

* PLANILHA_FINAL → dados tratados
* ERROS_FAIXA → inconsistências

> Status
* OK → correto
* PRECO_INCOERENTE → erro de faixa
* PRECO_TABELA → preenchido automaticamente
* SEM_PROMOCAO → sem faixa promocional
* PRECO_VAZIO → não encontrado na tabela

                                                               Observação

**O script não corrige preços automaticamente, apenas sinaliza para revisão.**

In [2]:
import pandas as pd
from google.colab import files

# Upload dos arquivos
uploaded = files.upload()
arquivos = list(uploaded.keys())

print("Arquivos enviados:")
for arq in arquivos:
    print("-", arq)

arquivo_campanha = input("Digite exatamente o nome do arquivo da campanha: ")
arquivo_tabela = input("Digite exatamente o nome do arquivo da tabela normal: ")

# Ler arquivos
df = pd.read_excel(arquivo_campanha)
df_tabela = pd.read_excel(arquivo_tabela)

# Colunas
col_codigo = "CODIGO"
col_descricao = "DESCRIÇÃO PRODUTO"
col_qtd_inicial = "QTD INICIAL"
col_qtd_final = "QTD FINAL"
col_preco_final = "PREÇO FINAL"

col_codigo_tabela = "CODIGO"
col_preco_tabela = "PREÇO TABELA"   # ajuste se necessário

# Padronizar tipos
df[col_codigo] = df[col_codigo].astype(str).str.strip()
df_tabela[col_codigo_tabela] = df_tabela[col_codigo_tabela].astype(str).str.strip()

df[col_qtd_inicial] = pd.to_numeric(df[col_qtd_inicial], errors="coerce")
df[col_qtd_final] = pd.to_numeric(df[col_qtd_final], errors="coerce")
df[col_preco_final] = pd.to_numeric(df[col_preco_final], errors="coerce")
df_tabela[col_preco_tabela] = pd.to_numeric(df_tabela[col_preco_tabela], errors="coerce")

# Ordenar a campanha antes de processar
df = df.sort_values(by=[col_codigo, col_qtd_inicial, col_qtd_final], ascending=[True, True, True]).reset_index(drop=True)

# Mapa de preço da tabela
mapa_preco = dict(zip(df_tabela[col_codigo_tabela], df_tabela[col_preco_tabela]))

linhas_saida = []

for codigo, grupo in df.groupby(col_codigo, sort=False):
    grupo = grupo.sort_values(by=[col_qtd_inicial, col_qtd_final], ascending=[True, True]).copy()

    # Verifica se existe item sem promoção: 0 a 999999
    sem_promocao = (
        (grupo[col_qtd_inicial] == 0) &
        (grupo[col_qtd_final] == 999999)
    ).any()

    # Se for item sem promoção, não cria linha extra
    if sem_promocao:
        for _, row in grupo.iterrows():
            linha = row.copy()
            linha[col_preco_final] = mapa_preco.get(codigo, pd.NA)
            linha["STATUS_FAIXA"] = "SEM_PROMOCAO"
            linhas_saida.append(linha)
        continue

    # Se tiver promoção, cria a faixa anterior usando a menor QTD INICIAL do código
    menor_qtd_inicial = grupo[col_qtd_inicial].min()

    if pd.notna(menor_qtd_inicial) and menor_qtd_inicial > 0:
        primeira_linha = grupo.iloc[0].copy()
        primeira_linha[col_qtd_inicial] = 0
        primeira_linha[col_qtd_final] = menor_qtd_inicial - 1
        primeira_linha[col_preco_final] = mapa_preco.get(codigo, pd.NA)
        primeira_linha["STATUS_FAIXA"] = "PRECO_TABELA"
        linhas_saida.append(primeira_linha)

    # Depois adiciona as linhas promocionais em ordem
    for _, row in grupo.iterrows():
        linha_original = row.copy()
        linha_original["STATUS_FAIXA"] = "PENDENTE_VALIDACAO"
        linhas_saida.append(linha_original)

# Criar dataframe final
df_saida = pd.DataFrame(linhas_saida)

# Ordenar resultado final
df_saida = df_saida.sort_values(
    by=[col_codigo, col_qtd_inicial, col_qtd_final],
    ascending=[True, True, True]
).reset_index(drop=True)

# Validar coerência das faixas promocionais
status_corrigido = []

for codigo, grupo in df_saida.groupby(col_codigo, sort=False):
    grupo = grupo.sort_values(by=[col_qtd_inicial, col_qtd_final]).copy()
    preco_anterior = None

    for _, row in grupo.iterrows():
        status = row["STATUS_FAIXA"]
        preco = row[col_preco_final]

        if status in ["SEM_PROMOCAO", "PRECO_TABELA"]:
            status_corrigido.append(status)
            continue

        if pd.isna(preco):
            status_corrigido.append("PRECO_VAZIO")
            continue

        if preco_anterior is None:
            status_corrigido.append("OK")
            preco_anterior = preco
        else:
            if preco <= preco_anterior:
                status_corrigido.append("OK")
            else:
                status_corrigido.append("PRECO_INCOERENTE")
            preco_anterior = preco

df_saida["STATUS_FAIXA"] = status_corrigido

# Aba de erros
df_erros = df_saida[df_saida["STATUS_FAIXA"] == "PRECO_INCOERENTE"].copy()

# Salvar
nome_saida = "VENDA_MAIS_ORDENADA.xlsx"
with pd.ExcelWriter(nome_saida, engine="openpyxl") as writer:
    df_saida.to_excel(writer, index=False, sheet_name="PLANILHA_FINAL")
    df_erros.to_excel(writer, index=False, sheet_name="ERROS_FAIXA")

files.download(nome_saida)

Saving tabela neo.xlsx to tabela neo (1).xlsx
Saving venda mais.xlsx to venda mais (1).xlsx
Arquivos enviados:
- tabela neo (1).xlsx
- venda mais (1).xlsx
Digite exatamente o nome do arquivo da campanha: venda mais.xlsx
Digite exatamente o nome do arquivo da tabela normal: tabela neo.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>